# tinylm sft — analysis

Analysis-only notebook: loads the SFT checkpoints produced by `train.py`
(full-FT and, when present, LoRA) and compares them against the base pretrain
checkpoint — chat generations, per-source masked val loss, zero-shot benchmarks
(SFT shouldn't tank them). Run `train.py` (and `USE_LORA=1 train.py`) first.


In [ ]:
import os

os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")

from pathlib import Path

import torch

import train  # sft/train.py: constants, make_model, make_datamodules

DEVICE = train.DEVICE

CKPTS = {
    "base": Path(train.BASE_CHECKPOINT),
    "full-ft": train.RUN_DIR / "chimera_gpt6m_sft.pt",
    "lora-r16": train.RUN_DIR / "chimera_gpt6m_sft_lora.pt",
}
CKPTS = {k: p for k, p in CKPTS.items() if p.exists()}

dms = train.make_datamodules()
tok = dms[0].tokenizer
eos_id, bos_id = dms[0].eos_id, dms[0].bos_id
im_end_id = tok._tok.token_to_id("<|im_end|>")

def load(name):
    m = train.make_model(tok.vocab_size)
    m.load_state_dict(torch.load(CKPTS[name], map_location="cpu"))
    return m.to(DEVICE, torch.bfloat16).eval()

models = {name: load(name) for name in CKPTS}
print("loaded:", ", ".join(models))


## Masked val loss per source

Loss over assistant tokens only, per SFT source. `base` shows where each
checkpoint started; deltas show what SFT actually moved.


In [ ]:
import pandas as pd

rows = {}
for name, m in models.items():
    rows[name] = {
        dm.name: train.evaluate(m, dm.val_dataloader(), eos_id) for dm in dms
    }
pd.DataFrame(rows).round(4)


## Chat


In [ ]:
from chimera.data.chat_template import render

@torch.no_grad()
def chat(model, question, max_new_tokens=80):
    prompt = render([{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tok._tok.encode(prompt, add_special_tokens=False).ids
    x = torch.tensor([[bos_id] + ids], device=DEVICE)
    out = []
    for _ in range(max_new_tokens):
        nxt = int(model(x)[0, -1].argmax())
        if nxt in (eos_id, im_end_id):
            break
        out.append(nxt)
        x = torch.cat([x, torch.tensor([[nxt]], device=DEVICE)], dim=1)
    return tok._tok.decode(out).strip()

PROMPTS = [
    "What color is the sky?",
    "Hi! How are you today?",
    "Tom has a red ball and a blue kite. He plays with them in the park every "
    "Sunday with his sister Amy.\n\nWhat color is Tom's ball?",
]

for q in PROMPTS:
    print(f"### {q[:70]}")
    for name, m in models.items():
        print(f"  {name:>8}: {chat(m, q)!r}")
    print()


## Zero-shot benchmarks

Same task set as pretrain. The check is that SFT didn't tank the base
capabilities (catastrophic forgetting), not that it improves them.


In [ ]:
from chimera.evals import ChimeraLM, TASKS, headline, run_eval

bench = {}
for name, m in models.items():
    lm = ChimeraLM(m, tok, eot_id=eos_id, bos_id=bos_id,
                   block_size=m.seq_len, device=DEVICE, batch_tokens=131_072)
    results = run_eval(lm, TASKS)
    bench[name] = {t: headline(results[t])[1] * 100 for t in TASKS}
pd.DataFrame(bench).round(2)


Past results and run notes live in the [README](README.md).
